# Deploy to Neuromorphic Hardware

**Nuro SDK** — Train on GPU, deploy to silicon. Zero code changes.

In this notebook, you'll:
1. Build and train an SNN on GPU
2. Save the trained weights
3. Deploy to Intel Loihi 2 (simulator)
4. Deploy to SpiNNaker 2 (simulator)
5. Deploy to BrainChip Akida
6. Use NIR to import models from other frameworks

All hardware backends use **simulators** — you don't need physical chips to follow along.

[![GitHub](https://img.shields.io/badge/GitHub-Vantar--AI%2Fnuro-black)](https://github.com/Vantar-AI/nuro)
[![License](https://img.shields.io/badge/License-Apache%202.0-blue)](https://github.com/Vantar-AI/nuro/blob/main/LICENSE)

## 1. Install

Install Nuro with GPU support. Hardware backends are optional extras.

In [ ]:
# Core + GPU (always works)
!pip install -q nuro[gpu]

# Optional: hardware backends (install if you have access)
# !pip install nuro[loihi]      # Intel Loihi 2 (requires lava-nc, Python 3.10)
# !pip install nuro[spinnaker2] # SpiNNaker 2 (requires py-spinnaker2)
# !pip install nuro[akida]      # BrainChip Akida (requires akida SDK)
# !pip install nuro[nir]        # NIR interop (cross-framework models)

## 2. Train on GPU

First, let's build and train a small SNN. This is our "development workbench."

In [ ]:
import torch
import nuro

print(f"Nuro {nuro.__version__}")

# Define the network
inp_pop = nuro.Population(size=100, dynamics="lif", params={"tau": 20e-3})
hidden  = nuro.Population(size=50,  dynamics="lif", params={"tau": 15e-3})
motor   = nuro.Population(size=10,  dynamics="lif", params={"tau": 10e-3})

c1 = nuro.Connection(source=inp_pop, target=hidden, pattern="dense")
c2 = nuro.Connection(source=hidden,  target=motor,  pattern="dense")

# Create input data (Poisson-like spike trains)
num_steps = 50
spike_data = (torch.rand(num_steps, 100) < 0.1).float()
inp = nuro.Input(population=inp_pop, data=spike_data)

graph = nuro.Graph([inp_pop, hidden, motor], [c1, c2], inputs=[inp])

# Train on GPU
model = nuro.compile(graph, target="gpu", requires_grad=True, surrogate="atan")
optimizer = torch.optim.Adam(model.snn.parameters(), lr=1e-3)

# Simple training loop (for demonstration)
for step in range(20):
    optimizer.zero_grad()
    output = model.run(duration=0.05, dt=1e-3)
    # Encourage motor neuron 0 to fire the most
    target_spikes = torch.zeros(10)
    target_spikes[0] = 5.0
    loss = ((output[motor.id] - target_spikes) ** 2).mean()
    loss.backward()
    optimizer.step()
    if step % 5 == 0:
        print(f"Step {step:3d} | Loss: {loss.item():.4f} | Spikes: {output[motor.id].detach().numpy().round(1)}")

print("\nTraining complete!")

## 3. Save Trained Weights

Save the GPU checkpoint. This file contains all synaptic weights that Nuro will transfer to hardware backends.

In [ ]:
model.save("trained_snn.pt")
print("Saved: trained_snn.pt")

# Inspect what's inside
checkpoint = torch.load("trained_snn.pt", weights_only=True)
for key, tensor in checkpoint.items():
    if 'weight' in key or 'bias' in key:
        print(f"  {key}: {tensor.shape}")

## 4. Deploy to Intel Loihi 2

Nuro compiles your graph to **Lava** (Intel's neuromorphic SDK) — you never write Lava code directly.

Weights are automatically quantized to 8-bit integers matching Loihi 2's fixed-point format.

In [ ]:
try:
    # Compile to Loihi 2 with trained weights
    loihi_model = nuro.compile(
        graph,
        target="loihi",
        weights_from="trained_snn.pt",
    )

    # Run on Loihi simulator
    output = loihi_model.run(duration=0.05)
    print("Loihi 2 deployment successful!")
    print(f"Output spikes: {output}")

except ImportError:
    print("lava-nc not installed. Install with: pip install nuro[loihi]")
    print("Note: lava-nc requires Python 3.10")
    print("")
    print("What happens under the hood:")
    print("  1. Nuro builds a Lava Process graph (LIF + Dense)")
    print("  2. GPU weights are quantized to 8-bit integers")
    print("  3. Weights are loaded into Lava Dense connections")
    print("  4. Network runs on Loihi 2 simulator (or real chip via INRC)")

## 5. Deploy to SpiNNaker 2

Same network, different target. Nuro maps to **py-spinnaker2** and Brian2 simulator.

In [ ]:
try:
    # Compile to SpiNNaker 2
    s2_model = nuro.compile(
        graph,
        target="spinnaker2",
        weights_from="trained_snn.pt",
    )

    output = s2_model.run(duration=0.05)
    print("SpiNNaker 2 deployment successful!")

except ImportError:
    print("py-spinnaker2 not installed. Install with: pip install nuro[spinnaker2]")
    print("")
    print("What happens under the hood:")
    print("  1. Nuro creates Brian2 neuron groups with SpiNNaker 2 equations")
    print("  2. Weights are converted to connection lists")
    print("  3. Runs on Brian2 simulator or SpiNNcloud hardware")

## 6. Deploy to BrainChip Akida

Akida is the most commercially deployed neuromorphic chip. Nuro compiles to the MetaTF SDK.

In [ ]:
try:
    # Compile to Akida
    akida_model = nuro.compile(
        graph,
        target="akida",
        weights_from="trained_snn.pt",
    )

    output = akida_model.run(duration=0.05)
    print("Akida deployment successful!")

except ImportError:
    print("akida SDK not installed. Install with: pip install nuro[akida]")
    print("")
    print("What happens under the hood:")
    print("  1. Nuro maps LIF neurons to Akida InputConvolutional/Dense layers")
    print("  2. Weights quantized to 1-8 bit (configurable)")
    print("  3. Deploys to Akida 1.0 (AKD1000) or 2.0 (AKD1500)")

## 7. Side-by-Side Comparison

The key insight: **your network definition never changes**. Only the compile target does.

In [ ]:
print("""
# Your network definition (IDENTICAL for all backends)
inp = nuro.Population(size=100, dynamics="lif")
hid = nuro.Population(size=50,  dynamics="lif")
out = nuro.Population(size=10,  dynamics="lif")
c1  = nuro.Connection(source=inp, target=hid, pattern="dense")
c2  = nuro.Connection(source=hid, target=out, pattern="dense")

# Only this line changes:
model = nuro.compile(graph, target="gpu")         # Development
model = nuro.compile(graph, target="loihi")       # Intel Loihi 2
model = nuro.compile(graph, target="spinnaker2")  # SpiNNaker 2
model = nuro.compile(graph, target="akida")       # BrainChip Akida

# Comparison:
#                    GPU          Loihi 2       SpiNNaker 2    Akida
# Power:           ~250W         ~1W           ~1W            ~300mW
# Weights:         float32       int8          int16          1-8 bit
# Use case:        Training      Edge deploy   Large-scale    Commercial edge
# Nuro backend:    PyTorch       Lava          py-spinnaker2  MetaTF
""")

## 8. Advanced: New v0.7 Features

### Synaptic Delays

Add biologically realistic propagation delays to connections.

In [ ]:
# Synaptic delays — spikes arrive after a fixed delay
c1_delayed = nuro.Connection(
    source=inp_pop,
    target=hidden,
    pattern="dense",
    delay=2e-3,  # 2ms propagation delay
)

print(f"Connection delay: {c1_delayed.delay * 1000}ms")
print("GPU: ring buffer implementation")
print("Loihi: native hardware delay support")
print("SpiNNaker: connection_list delay column")

### NIR Interop

Import models from SpikingJelly, Norse, snnTorch, or any NIR-compatible framework.

In [ ]:
# NIR (Neuromorphic Intermediate Representation) interop
# Import from any NIR-compatible framework:

print("""
# Import a model from snnTorch/Norse/SpikingJelly via NIR:
import nir
nir_graph = nir.read("model_from_snntorch.nir")
nuro_graph = nuro.from_nir(nir_graph)
model = nuro.compile(nuro_graph, target="loihi")  # Deploy to hardware!

# Export a Nuro model to NIR (for use in other frameworks):
nir_graph = nuro.to_nir(graph)
nir.write("my_model.nir", nir_graph)

# NIR is supported by 9 frameworks + 5 hardware platforms.
# Nuro is now part of this ecosystem.
""")

### Connectivity Patterns

In [ ]:
# New connectivity patterns in v0.7

# One-to-one (diagonal) — each neuron connects to exactly one target
p1 = nuro.Population(size=50, dynamics="lif")
p2 = nuro.Population(size=50, dynamics="lif")
c_121 = nuro.Connection(source=p1, target=p2, pattern="one_to_one")

# 1D Convolution — sliding kernel connectivity
c_conv = nuro.Connection(
    source=p1, target=p2, pattern="conv1d",
    params={"kernel_size": 5, "stride": 1}
)

# Distance-dependent — connection probability falls off with distance
c_dist = nuro.Connection(
    source=p1, target=p2, pattern="distance_dependent",
    params={"sigma": 10.0}
)

print("Available patterns: dense, random_sparse, one_to_one, conv1d, distance_dependent")

### Training Callbacks

In [ ]:
from nuro.callbacks import PrintCallback

# Run with progress logging
model = nuro.compile(graph, target="gpu")
output = model.run(
    duration=0.05,
    dt=1e-3,
    callbacks=[PrintCallback(every_n_steps=10)]
)

print(f"\nTotal spikes: {model.metrics['total_spikes']}")
print("")
print("Other callbacks available:")
print("  WandbCallback(project='my-snn')    # Weights & Biases logging")
print("  TensorBoardCallback(log_dir='runs') # TensorBoard logging")

## Summary

| Backend | Status | Hardware | Auto-Quantize |
|---------|--------|----------|---------------|
| `gpu` | Stable | Any CUDA GPU | N/A |
| `loihi` | Stable | Loihi 2 (sim or INRC) | 8-bit |
| `spinnaker2` | Stable | SpiNNaker 2 (sim or SpiNNcloud) | 16-bit |
| `akida` | Stable | Akida 1.0/2.0 | 1-8 bit |
| `cloud` | Beta | Remote hardware via API | Auto |

**The entire workflow:**
1. Define your SNN (populations, connections)
2. Train on GPU with surrogate gradients
3. `model.save("weights.pt")`
4. `nuro.compile(graph, target="loihi", weights_from="weights.pt")`
5. Done. 1000x more energy efficient at inference.

---

[GitHub](https://github.com/Vantar-AI/nuro) | [Website](https://vantar.xyz) | [Docs](https://github.com/Vantar-AI/nuro/tree/main/docs)